## Setup

Usage: Before running any cells, make sure the config within the Setup block is correct.  Mark the correct PORT and BAUD if connected with USB.  Mark LOAD_FROM_FILE as True and paste the data file paths you wish to use if not.  Then, run the Estimate Sampling Rate and Carrier Frequency block to fill in the correct sampling rate and carrier frequency based on the data.  Any block can then be ran, make sure the correct amount of data files are provided since some blocks use several data files if marked as so in their respective configs.

In [ ]:
import serial
import time
import contextlib
from datetime import datetime
from pathlib import Path
from scipy.signal import get_window
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.signal as signal
import numpy.fft as fft
import pyqtgraph as pg
from pyqtgraph.Qt import QtCore
from scipy.signal import welch
import padasip as pa

### CONFIG

PORT = "COM8"
BAUD = 921600
#Ensure all data files have the same sampling rate and carrier freq.

LOAD_FROM_FILE = False
LOAD_FILE_PATHS= [
    Path(r"data\averaging_data\07_20_2026\run_13_12_24\data_A2V-16_1.csv"),
    Path(r"data\averaging_data\07_20_2026\run_13_12_24\data_A2V-16_2.csv"),
    Path(r"data\averaging_data\07_20_2026\run_13_12_24\data_A2V-16_3.csv"),
    Path(r"data\averaging_data\07_20_2026\run_13_12_24\data_A2V-16_4.csv"),
    ]

SENSORS = ['A2V-16_A1', 'A2V-16_A2', 'A2V-16_A3', 'A2V-16_A4']
ADC_MAX = 16383 #2^14 - 1
V_REF = 5
SAMPLES_PER_PACKET = 50

###

if not LOAD_FROM_FILE:
    LOAD_FILE_PATHS = []

payload_dtype = np.dtype([
    ('timestamp', '<u4'),
    ('A2V-16_A1', '<u2'),
    ('A2V-16_A2', '<u2'),
    ('A2V-16_A3', '<u2'),
    ('A2V-16_A4', '<u2')
])

BYTES_PER_SAMPLE = payload_dtype.itemsize * SAMPLES_PER_PACKET
ser = None
reader = None

def fir_lowpass_filter(data, cutoff_hz, fs, num_taps=101):
    b = signal.firwin(num_taps, cutoff=cutoff_hz, fs=fs, window='hamming')
    filtered_data = np.asarray(signal.lfilter(b, 1.0, data))
    return filtered_data, b

def adc_to_mv(adc_value):
    return (adc_value / ADC_MAX) * V_REF * 1000  # Convert to millivolts

def fmt_matrix(M, title, fmt="{:>12.4f}"):
    s = f"\n{title}\n"
    s += " " * 10 + "".join(f"{ch:>12}" for ch in SENSORS) + "\n"
    for i, ch in enumerate(SENSORS):
        s += f"{ch:>10}" + "".join(fmt.format(M[i, j]) for j in range(len(SENSORS))) + "\n"
    return s

# reads one 50-sample packet from serial.
# Fix 3: resync by scanning to the next 0xAA 0xBB frame boundary with read_until, so a
# misaligned/corrupt stream self-realigns instead of blindly jumping a fixed 602 bytes
# (which could step past the real boundary forever on false-locks inside the payload).
def try_read_realtime(ser):
    sync = ser.read_until(b'\xAA\xBB')          # discard junk up to the next frame boundary
    if not sync.endswith(b'\xAA\xBB'):
        return None                             # timed out before a sync word appeared
    raw_data = ser.read(BYTES_PER_SAMPLE)
    if len(raw_data) != BYTES_PER_SAMPLE:
        return None
    data = np.frombuffer(raw_data, dtype=payload_dtype)
    dts = np.diff(data['timestamp'].astype(np.int64))
    ok = (all((data[s] <= ADC_MAX).all() for s in SENSORS) and
          (dts > 0).all() and (dts < 5000).all())    # rejects false locks inside payload data
    return data if ok else None

# read 50 samples from file to emulate realtime reading.
def read_from_file_realtime(file_path):
    df = pd.read_csv(file_path)
    data = df[list(payload_dtype.names or ())].to_records(index=False).astype(payload_dtype)
    n_packets = len(data) // SAMPLES_PER_PACKET
    packets = [data[i * SAMPLES_PER_PACKET:(i + 1) * SAMPLES_PER_PACKET]
               for i in range(n_packets)]

    idx = 0
    ts_start = int(packets[0]['timestamp'][0]) if packets else 0 
    wall_start = None 

    def reader(*_): 
        nonlocal idx, wall_start
        if idx >= len(packets):
            return np.zeros(SAMPLES_PER_PACKET, dtype=payload_dtype)  # exhausted
        pkt = packets[idx]
        now = time.time()
        if wall_start is None:  # start the clock on the first call
            wall_start = now
        # only release the packet once real time has caught up to its timestamp
        elapsed_us = (now - wall_start) * 1e6
        if elapsed_us < int(pkt['timestamp'][0]) - ts_start:
            return None
        idx += 1
        return pkt

    return reader

def read_from_file_instant(file_path):
    df = pd.read_csv(file_path)
    data = df.to_records(index=False).astype(payload_dtype)
    return data

def get_save_dir(base_folder: str, sample_count: int) -> Path:
    now = datetime.now()
    date_str = now.strftime('%m_%d_%Y') 
    time_str = now.strftime('%H_%M_%S')
    save_dir = Path("data") / base_folder / date_str / f"run_{sample_count}_samples_{time_str}"
    save_dir.mkdir(parents=True, exist_ok=True)
    return save_dir

def try_initialize_serial():
    if not LOAD_FROM_FILE:
        global ser
        if ser is not None and ser.is_open:
            ser.close()
        ser = serial.Serial(PORT, BAUD, timeout=1)
        ser.reset_input_buffer()    # Fix 2: drop the stale free-running backlog on open
        ser.reset_output_buffer()

def try_close_serial():
    global ser
    if ser is not None and ser.is_open:
        ser.close()

def drain_to_latest(ser):
    # Fix 2: discard any buffered backlog so capture starts on fresh, frame-aligned data.
    if ser is not None:
        ser.reset_input_buffer()

@contextlib.contextmanager
def serial_port():
    # Fix 1: guaranteed-close port. Use `with serial_port() as ser:` so the handle is
    # released even on KeyboardInterrupt, a thrown exception, or a closed plot window --
    # no more stuck COM port that needs an unplug/replug. Sets the global `ser` so any
    # code that still references it keeps working. Yields None when replaying from file.
    global ser
    if LOAD_FROM_FILE:
        yield None
        return
    ser = serial.Serial(PORT, BAUD, timeout=1)
    try:
        ser.reset_input_buffer()    # Fix 2: open flushed
        ser.reset_output_buffer()
        yield ser
    finally:
        ser.close()

def save_summary(save_dir, text, filename="summary_verbose.txt"):
    if save_dir is None:
        return None
    path = Path(save_dir) / filename
    path.write_text(text, encoding="utf-8")
    print(f"Saved {filename} to {path.as_posix()}")
    return path

def save_run_data(base_folder: str, data, filenames, sample_count, summary: str = ""):
    if LOAD_FROM_FILE:
        return None
    save_dir = get_save_dir(base_folder, sample_count)

    if len(data) != len(filenames):
        raise ValueError("Length of data and filenames must match.")
    for d, filename in zip(data, filenames):
        if d is not None:
            csv_path = save_dir / filename
            df = pd.DataFrame(d)
            df.to_csv(csv_path, index=False)
            print(f"Saved {len(d)} samples to {csv_path.as_posix()}")
        
    if summary:
        summary_path = save_dir / "summary.txt"
        summary_path.write_text(summary, encoding="utf-8")
        print(f"Saved summary to {summary_path.as_posix()}")
        
    return save_dir

def read_data_realtime(samples_total):
    # Fix 1: `with serial_port()` guarantees the port closes on any exit path.
    # Fix 2: opens flushed + drains before the loop.  Fix 3: try_read_realtime resyncs.
    chunks = []
    sample_count = 0
    try:
        with serial_port() as ser:
            drain_to_latest(ser)
            last_data = time.time()
            while samples_total is None or sample_count < samples_total:
                data = try_read_realtime(ser)
                if data is None:
                    if time.time() - last_data > 5:
                        print("No data for 5 s -- stopping capture.")
                        break
                    continue
                if np.mean(np.diff(data['timestamp'])) <= 0:
                    print("Timestamps did not change")
                    continue
                last_data = time.time()   # reset idle clock on a good packet
                chunks.append(data)
                sample_count += SAMPLES_PER_PACKET
    except KeyboardInterrupt:
        print("Stopping...")
    if not chunks:
        print("No data captured.")
        return np.empty(0, dtype=payload_dtype)   # always an ndarray -> no Optional warnings
    return np.concatenate(chunks)



## Estimate Sampling Rate and Carrier Frequency

This must be once after opening the notebook and when changing sampling rate or carrier frequency

In [2]:
### CONFIG

SAMPLES_TO_CAPTURE = 20000  # total number of samples to capture

###

data = {}
Fs = 1500.0  # initial guess for sampling rate
CARRIER_FREQ = 0  # initial guess for carrier frequency in Hz before estimation

data = read_from_file_instant(LOAD_FILE_PATHS[0]) if LOAD_FROM_FILE else read_data_realtime(SAMPLES_TO_CAPTURE)
data = data[:SAMPLES_TO_CAPTURE]  # limit to the first N samples

Fs = float(np.mean([1.0 / (np.median(np.diff(data['timestamp'])) * 1e-6) for i in range(len(data['timestamp']))]))
estimated_carrier_freqs = {}
for s in SENSORS:
    x = adc_to_mv(data[s])
    fft_vals = fft.fft(x)
    f = fft.fftfreq(len(x), d=1/Fs)
    mags = 2 * np.abs(fft_vals) / len(x)
    inband = (f > 14)  # skip the DC / drift / 1-f region
    peaks, _ = signal.find_peaks(mags[inband], distance=5, height=0.1)  # adjust height threshold as needed
    sorted_peaks = sorted(peaks, key=lambda p: mags[inband][p], reverse=True)
    estimated_carrier_freqs[s] = f[inband][sorted_peaks[0]] if sorted_peaks else 0.0

CARRIER_FREQ = np.mean(list(estimated_carrier_freqs.values()))
print(f"Estimated sampling rate: {Fs:.2f} Hz")
print(f"Average estimated carrier frequency across sensors: {CARRIER_FREQ:.6f} Hz")

Estimated sampling rate: 5000.00 Hz
Average estimated carrier frequency across sensors: 930.000000 Hz


## Saves realtime data to file


In [ ]:
NUM_SAMPLES_TO_SAVE = 3000
NUM_TRIALS = 10
data = {}
if not LOAD_FROM_FILE:  
    d = read_data_realtime(NUM_SAMPLES_TO_SAVE * NUM_TRIALS)
    for i in range(NUM_TRIALS):
        data[i] = d[i * NUM_SAMPLES_TO_SAVE:(i + 1) * NUM_SAMPLES_TO_SAVE]

save_run_data("raw_data", data=[data[i] for i in range(NUM_TRIALS)], sample_count=NUM_SAMPLES_TO_SAVE, filenames=[f"data_{i}.csv" for i in range(NUM_TRIALS)])

## Realtime plot

Reads packets from the Arduino and draws them

In [3]:

### CONFIG

SAMPLE_MEMORY = 10000
FILTER = False # set to True to enable bandpass filtering
BANDWIDTH = 4  # Hz, width of the bandpass filter around the carrier frequency

###

assert SAMPLE_MEMORY % SAMPLES_PER_PACKET == 0
buffers = {s: np.full(SAMPLE_MEMORY, np.nan) for s in SENSORS} 
write_idx = 0
label_style = {'color': '#FFF', 'font-size': '16pt'}

app = pg.mkQApp("realtime")
win = pg.GraphicsLayoutWidget(show=True, title="realtime")
pen = pg.mkPen('r', width=5)
curves = {}
for i, s in enumerate(SENSORS):
    p = win.ci.addPlot(row=i, col=0)
    p.setLabel('left', 'mV', **label_style)
    p.setLabel('top', f'{s}', **label_style)
    curves[s] = p.plot(pen=pen, connect='finite')  # 'finite' leaves a gap at the nan cursor instead of a line across it

sos_bp = signal.butter(4, [CARRIER_FREQ - BANDWIDTH/2, CARRIER_FREQ + BANDWIDTH/2], btype='bp', fs=Fs, output='sos')
sos_zi = {s: signal.sosfilt_zi(sos_bp) for s in SENSORS}

if LOAD_FROM_FILE:
    reader = read_from_file_realtime(LOAD_FILE_PATHS[0])

def update():
    # Sweep display: each new packet overwrites the 50 samples at the cursor and
    # nothing else moves, so previously plotted values never change.
    global write_idx
    got_packet = False

    while True:
        if LOAD_FROM_FILE:
            data = reader()
            if data is None:
                break
        else:
            if ser is not None and ser.in_waiting < BYTES_PER_SAMPLE + 2:
                break
            data = try_read_realtime(ser)
            if data is None:
                continue 
        for s in SENSORS:
            if FILTER:
                cleaned, sos_zi[s] = signal.sosfilt(sos_bp, data[s], zi=sos_zi[s])
                buffers[s][write_idx:write_idx + SAMPLES_PER_PACKET] = adc_to_mv(cleaned)
            else:
                buffers[s][write_idx:write_idx + SAMPLES_PER_PACKET] = adc_to_mv(data[s])
        write_idx = (write_idx + SAMPLES_PER_PACKET) % SAMPLE_MEMORY
        got_packet = True
        if LOAD_FROM_FILE:
            break  
    if got_packet:
        for s in SENSORS:
            buffers[s][write_idx:write_idx + SAMPLES_PER_PACKET] = np.nan  # blank slot marks the cursor
            curves[s].setData(buffers[s])

timer = QtCore.QTimer()
timer.timeout.connect(update)
timer.start(10)
try:
    with serial_port():          # Fix 1: port always closed, even on window-close/interrupt
        if not LOAD_FROM_FILE:
            drain_to_latest(ser)
        pg.exec()
finally:
    timer.stop()  # don't leave a stale timer attached if the cell is re-run


## Spectrogram Plotting

In [ ]:
### CONFIG

NFFT = 4096        
WINDOWSIZE = 512
WINDOW = 'han'
OVERLAP = 90
FILTER = False
FOLDER = "spectrograms"
CAPTURE_TIME = 9  # seconds
BANDWIDTH = 4  # Hz, for bandpass filter around carrier frequency

###

NUM_SAMPLES_TO_SAVE = int(Fs * CAPTURE_TIME)
data = read_from_file_instant(LOAD_FILE_PATHS[0]) if LOAD_FROM_FILE else read_data_realtime(NUM_SAMPLES_TO_SAVE)

win = get_window(WINDOW, WINDOWSIZE)
noverlap = int(WINDOWSIZE * OVERLAP / 100)
sos_bp = signal.butter(4, [CARRIER_FREQ - BANDWIDTH/2, CARRIER_FREQ + BANDWIDTH/2], btype='bp', fs=Fs, output='sos') 
dir = save_run_data(FOLDER, data=[data], filenames=["data.csv"], sample_count=NUM_SAMPLES_TO_SAVE, summary=f"Sampling Rate = {Fs:.2f} Hz\nCarrier Frequency = {CARRIER_FREQ:.6f} Hz\n")

with plt.style.context('dark_background'):
    fig, axes = plt.subplots(4, 1, figsize=(19, 21), sharex=False, constrained_layout=True)
    fig.suptitle(f'Channel spectrograms, Sampling Rate = {Fs:.2f} Hz', fontsize=30)

    for ax, s in zip(axes, SENSORS):
        if FILTER:
            spec, freqs, t_bins, im = ax.specgram(signal.sosfilt(sos_bp, adc_to_mv(data[s])), Fs=Fs, NFFT=WINDOWSIZE, window=win, noverlap=noverlap, pad_to=NFFT, cmap='magma')
        else:
            spec, freqs, t_bins, im = ax.specgram(adc_to_mv(data[s]), Fs=Fs, NFFT=WINDOWSIZE, window=win, noverlap=noverlap, pad_to=NFFT, cmap='magma')
        ax.set_ylabel(f'{s}\n(Hz)', fontsize = 30)
        ax.set_ylim(0, Fs/2)
        cbar = fig.colorbar(im, ax=ax, pad=0.01)
        cbar.set_label('Power (dB)', fontsize = 30)

    axes[-1].set_xlabel('Time (s)', fontsize = 30)
    if dir is not None:
        plt.savefig(dir / "spectrograms.png", dpi=300)
    plt.show()



## Realtime PSD

In [6]:
### CONFIG

FILTER = False # set to True to enable bandpass filtering
SAMPLE_MEMORY = 2000
BANDWIDTH = 4  # Hz, width of the bandpass filter around the carrier frequency

###

buffers = {s: np.zeros(SAMPLE_MEMORY) for s in SENSORS}
if FILTER:
    sos_bp = signal.butter(4, [CARRIER_FREQ - BANDWIDTH/2, CARRIER_FREQ + BANDWIDTH/2], btype='bp', fs=Fs, output='sos')
    sos_zi = {s: signal.sosfilt_zi(sos_bp) for s in SENSORS}

app = pg.mkQApp("QRNG PSD realtime")
win = pg.GraphicsLayoutWidget(show=True, title="QRNG PSD realtime")
curves = {}
plots = []
for i, s in enumerate(SENSORS):
    p = win.ci.addPlot(row=0, col=i, title=s)
    p.setLabel('left', 'PSD (mV^2/Hz)')
    p.setLabel('bottom', 'Frequency (Hz)')
    p.setLogMode(x=False, y=True)              # log Y: carrier + floor both visible, DC no longer blows up the scale
    p.getViewBox().setXRange(0, Fs / 2, padding=0)   # 0 -> Nyquist, tracks whatever Fs is

    plots.append(p)
    curves[s] = p.plot()

if LOAD_FROM_FILE:
    reader = read_from_file_realtime(LOAD_FILE_PATHS[0])

def update():
    if LOAD_FROM_FILE:
        data = reader()
    else:
        data = try_read_realtime(ser)
    if data is not None:
        for s in SENSORS:
            chunk_mv = adc_to_mv(data[s])
            buffers[s] = np.roll(buffers[s], -SAMPLES_PER_PACKET)
            if FILTER:
                chunk, sos_zi[s] = signal.sosfilt(sos_bp, chunk_mv, zi=sos_zi[s])
                buffers[s][-SAMPLES_PER_PACKET:] = chunk
            else:
                buffers[s][-SAMPLES_PER_PACKET:] = chunk_mv
            f, pxx = welch(buffers[s], fs=Fs, nperseg=512)
            curves[s].setData(f, pxx)

timer = QtCore.QTimer()
timer.timeout.connect(update)
timer.start()
try:
    with serial_port():          # Fix 1: port always closed, even on window-close/interrupt
        if not LOAD_FROM_FILE:
            drain_to_latest(ser)
        pg.exec()
finally:
    timer.stop()


## Variance of A Single Sensor over Sequential Time Periods

In [ ]:

### EDIT

NUM_SAMPLES_TO_SAVE = 50000   # set to None to run until interrupted
FOLDER = "temporal_variance_data"
SENSOR = SENSORS[0]
NUM_TRIALS = 2
DEMOD_LP_HZ = 20  # Hz, low-pass filter cutoff for demodulated signal
TRANSIENT = 200  # samples to drop to ensure filter settles
SETTLE_TAU = 7.0  # filter time constants to drop at the start (7*tau ~= -60 dB ring-down)
BANDWIDTH = 4  # Hz, width of the bandpass filter around the carrier frequency

###

PERIODS = [f"t{i+1}" for i in range(NUM_TRIALS)]
if LOAD_FROM_FILE and len(LOAD_FILE_PATHS) != NUM_TRIALS:
    raise ValueError(f"LOAD_FILE_PATHS must contain {NUM_TRIALS} file paths for analysis.")

data = {}   
if LOAD_FROM_FILE:
    data = {i: read_from_file_instant(LOAD_FILE_PATHS[i]) for i in range(NUM_TRIALS)}
else:
    d = read_data_realtime(NUM_SAMPLES_TO_SAVE * NUM_TRIALS)
    for i in range(NUM_TRIALS):
        data[i] = d[i * NUM_SAMPLES_TO_SAVE:(i + 1) * NUM_SAMPLES_TO_SAVE]


TRANSIENT_BP = int(np.ceil(SETTLE_TAU / (np.pi * BANDWIDTH) * Fs))
TRANSIENT_LP = int(np.ceil(SETTLE_TAU / (np.pi * DEMOD_LP_HZ) * Fs))
TRANSIENT_BP = int(np.ceil(TRANSIENT_BP / 1000) * 1000)  # round up to nearest 1000 samples
TRANSIENT_LP = int(np.ceil(TRANSIENT_LP / 1000) * 1000)  # round up to nearest 1000 samples


data_1sensor = {i: data[i][SENSOR] for i in range(NUM_TRIALS)}
        
sos_bp = signal.butter(4, [CARRIER_FREQ - BANDWIDTH/2, CARRIER_FREQ + BANDWIDTH/2], btype='bp', fs=Fs, output='sos')
sos_lp = signal.butter(4, DEMOD_LP_HZ, btype='low', fs=Fs, output='sos')

filtered_data = {i: signal.sosfiltfilt(sos_bp, data_1sensor[i]) for i in range(NUM_TRIALS)}
kept_filtered_data = {i: filtered_data[i][TRANSIENT_BP:-TRANSIENT_BP] for i in range(NUM_TRIALS)}


n = min(len(kept_filtered_data[i]) for i in range(NUM_TRIALS))
t = np.arange(n) / Fs 

A = {}
a = {}
for i in range(NUM_TRIALS):
    x = kept_filtered_data[i][:n]
    I = np.cos(2 * np.pi * CARRIER_FREQ * t) * x
    Q = -np.sin(2 * np.pi * CARRIER_FREQ * t) * x
    I_f = signal.sosfiltfilt(sos_lp, I)
    Q_f = signal.sosfiltfilt(sos_lp, Q)
    A[i] = (2 * np.sqrt(I_f**2 + Q_f**2))[TRANSIENT_LP:-TRANSIENT_LP]
    a[i] = np.mean(A[i])

a_vec = np.array([a[i] for i in range(NUM_TRIALS)])

var_A = np.array([np.var(A[i]) for i in range(NUM_TRIALS)])  # mV^2, total fluctuation per window
std_A = np.sqrt(var_A) # mV
frac  = std_A / a_vec # std/mean: fractional fluctuation (m + n combined)

result = f"A is the message encoded in the carrier wave (the total noise signal, LED FLUCTUATION + CIRCUIT NOISE + INTERFERENCE...)\n" 
result += f"a is the mean of the demodulated message\n"
result += f"Var(A) is the total variance of A\n"
result += f"std(A) is the standard deviation of A\n"
result += f"std/a is the fractional fluctuation (std/mean) for each time window.\n"

result += f"\nSame sensor ({SENSOR}) over {NUM_TRIALS} sequential time windows, Fs = {Fs:.2f} Hz\n"
result += f"{'window':>8}{'a (mV)':>12}{'Var(A) mV^2':>14}{'std(A) mV':>12}{'std/a':>12}\n"
for i in range(NUM_TRIALS):
    result += f"{PERIODS[i]:>8}{a_vec[i]:>12.4f}{var_A[i]:>14.4f}{std_A[i]:>12.4f}{frac[i]:>12.4e}\n"

# across-window summary: the spread of Var(A) shows how stationary the total
# fluctuation is. CV = std/mean of Var(A) across windows; small => stationary.
mean_var = var_A.mean()
std_var  = var_A.std(ddof=1)
cv = std_var / mean_var

print(result)
result += f"\nVar(A) across windows: E[Var(A)] = {mean_var:.4f} mV^2, std = {std_var:.4f} mV^2, spread (CV) = {100*cv:.1f}%"

data_to_save = [data[i] for i in range(NUM_TRIALS)]
file_names = [f"data_{PERIODS[i]}.csv" for i in range(NUM_TRIALS)]
save_run_data(FOLDER, data=data_to_save, filenames=file_names, sample_count=NUM_SAMPLES_TO_SAVE, summary=result)

## Find the Variance Between Sensors

$f_c$ = 500Hz

The measured voltage over the load resistor due to the current of a photodiode in response to an LED at $f_c$: $V(t) = A(t)cos(2 \pi f_c t + \theta) $  

$A(t) = a + am(t) + n(t)$ where

a = the mean of the carrier amplitude, the voltage due to photocurrent, mV 

m(t) = the LED relative intensity fluctuation (common to all of the PDS), dimensionless, E[m] = 0

n(t) = the indepedent noise of each PD (shot, thermal, etc) within the band, mV

By using IQ Demodulation, A(t) can be recovered from the carrier wave.

The covariance of all of the A signal pairs from each diode pair can be found as 

$Cov(A_1, A_2) = a_1a_2Var(m)$ since n(t) is uncorrelated and all PDs see the same fractional fluctuation from the LED.  Thus, $Var(m)$ can be isolated as $Var(m) = \frac{Cov(A_1,A_2)}{a_1a_2}$.



Var(m) can be estimated by using the above formula and taking the mean of the most correlated pairs.

Since n(t) is uncorrelated, it can be found that $Var(n_i) = Var(A_i) - a_i^2Var(m)$ for each PD.

In [ ]:
### CONFIG

NUM_SAMPLES_TO_SAVE = 200000
NUM_TRIALS = 2
CORR_GATE = 0.5      # only average pairs that actually share the common mode
FOLDER = "cross_sensor_variance_data"
BANDWIDTH = 4        # Hz, width of the bandpass filter around the carrier frequency
DEMOD_LP_HZ = 20    # Hz, demod low-pass cutoff
SETTLE_TAU = 7.0    

###

PERIODS = [f"t{i+1}" for i in range(NUM_TRIALS)]
if LOAD_FROM_FILE and len(LOAD_FILE_PATHS) != NUM_TRIALS:
    raise ValueError(f"LOAD_FILE_PATHS must contain {NUM_TRIALS} file paths for analysis.")

data = {}
if LOAD_FROM_FILE:
    data = {i: read_from_file_instant(LOAD_FILE_PATHS[i]) for i in range(NUM_TRIALS)}
else:
    d = read_data_realtime(NUM_SAMPLES_TO_SAVE * NUM_TRIALS)
    for i in range(NUM_TRIALS):
        data[i] = d[i * NUM_SAMPLES_TO_SAVE:(i + 1) * NUM_SAMPLES_TO_SAVE]

# filters:
# sos_bp: isolate the carrier (CARRIER_FREQ +/- BW/2 Hz)
# sos_lp: extract the demodulated envelope after mixing the carrier down to baseband
# Both zero-phase (filtfilt)
sos_bp = signal.butter(4, [CARRIER_FREQ - BANDWIDTH/2, CARRIER_FREQ + BANDWIDTH/2], btype='bp', fs=Fs, output='sos')
sos_lp = signal.butter(4, DEMOD_LP_HZ, btype='low', fs=Fs, output='sos')
offdiag = ~np.eye(len(SENSORS), dtype=bool)   # mask that selects off-diagonal (cross) entries

TRANSIENT_BP = int(np.ceil(SETTLE_TAU / (np.pi * BANDWIDTH) * Fs))
TRANSIENT_LP = int(np.ceil(SETTLE_TAU / (np.pi * DEMOD_LP_HZ) * Fs))
TRANSIENT_BP = int(np.ceil(TRANSIENT_BP / 1000) * 1000)  # round up to nearest 1000 samples
TRANSIENT_LP = int(np.ceil(TRANSIENT_LP / 1000) * 1000)  # round up to nearest 1000 samples


result = "A_i is the demodulated message of sensor i (LED FLUCTUATION + CIRCUIT NOISE + INTERFERENCE...)\n"
result += "a_i is the mean carrier amplitude of sensor i\n"
result += "Cov(A_i, A_j) = a_i * a_j * Var(m)  ->  Var(m) = Cov(A_i, A_j) / (a_i * a_j)\n"
result += "Var(n_i) = Var(A_i) - a_i^2 * Var(m)  (independent per-diode noise)\n"
result += f"\nAcross {len(SENSORS)} sensors over {NUM_TRIALS} independent windows, Fs = {Fs:.2f} Hz, Carrier Frequency = {CARRIER_FREQ:.2f} Hz\n"

var_m_trials = []

for trial in range(NUM_TRIALS):
    # raw ADC -> mV -> bandpass around the carrier, then drop the filter transient
    bandpassed = {}
    for s in SENSORS:
        v_mv = adc_to_mv(data[trial][s])
        v_bp = signal.sosfiltfilt(sos_bp, v_mv)
        bandpassed[s] = v_bp[TRANSIENT_BP:-TRANSIENT_BP]

    n = min(len(bandpassed[s]) for s in SENSORS)   # common length across channels
    t = np.arange(n) / Fs

    # 1. Complex I/Q Downconversion
    lo_cos = np.cos(2 * np.pi * CARRIER_FREQ * t)
    lo_sin = -np.sin(2 * np.pi * CARRIER_FREQ * t)

    Z = {}
    for s in SENSORS:
        I = signal.sosfiltfilt(sos_lp, lo_cos * bandpassed[s][:n])
        Q = signal.sosfiltfilt(sos_lp, lo_sin * bandpassed[s][:n])
        Z[s] = I + 1j * Q

    # 2. De-rotate using highest-amplitude channel as phase reference
    ref = max(SENSORS, key=lambda s: np.abs(Z[s]).mean())
    phi = np.unwrap(np.angle(Z[ref]))
    slope, offset = np.polyfit(t, phi, 1)
    freq_err = slope / (2 * np.pi)  # Carrier frequency error in Hz

    # Apply phase correction vector across all channels
    sig = {s: Z[s] * np.exp(-1j * (slope * t + offset)) for s in SENSORS}

    # 3. Recover each channel's de-rotated envelope A_i(t) and mean amplitude a_i
    A = {}
    a = {}
    for s in SENSORS:
        envelope = 2.0 * np.abs(sig[s])
        A[s] = envelope[TRANSIENT_LP:-TRANSIENT_LP]  # drop low-pass edge transient
        a[s] = A[s].mean()

    X = np.vstack([A[s] for s in SENSORS])
    a_vec  = np.array([a[s] for s in SENSORS])
    cov_A  = np.cov(X) # mV^2, Var(A_i) on the diagonal
    corr_A = np.corrcoef(X)  # dimensionless, in [-1, 1]
    norm_cov = cov_A / np.outer(a_vec, a_vec) # Cov(A_i,A_j) / (a_i * a_j)

    # mask out entries that aren't correlated; var(m) = mean of what survives (nan if none)
    gated = np.where(offdiag & (corr_A > CORR_GATE), norm_cov, np.nan)
    var_m = np.nanmean(gated)
    n_pairs = int(np.isfinite(gated).sum() // 2)
    is_coupled = np.isfinite(gated).sum(axis=1) > 0
    var_m_trials.append(var_m)

    var_A = np.diag(cov_A)  # total envelope variance per channel
    var_common = a_vec ** 2 * var_m # portion explained by the shared LED
    var_n = var_A - var_common # remainder = independent, uncorrelated noise

    result += f"\n{'=' * 64}\nTRIAL {PERIODS[trial]}\n{'=' * 64}\n"
    result += f"LO frequency error df = {freq_err:+.4f} Hz (ref channel: {ref})\n\n"
    result += fmt_matrix(cov_A, "[1] COVARIANCE (mV^2) - diagonal = total fluctuations of each channel, off-diagonal = fluctuation shared between two channels")
    result += fmt_matrix(corr_A, "[2] CORRELATION (-1 to 1)- near 1 = channels track togther (shared LED), 0 = independent")
    result += fmt_matrix(norm_cov, "[3] NORMALIZED COVARIANCE - off-diagonals = var(m) estimate per pair; all equal => one shared LED fluctuation", fmt="{:>12.4e}")
    result += f"\nvar(m) = {var_m:.4e}   RMS(m) = sqrt(var(m)) = {np.sqrt(var_m):.4f}"
    result += f"   (mean over {n_pairs} coupled pair(s), corr > {CORR_GATE})\n"

    # per-channel noise split: Var(A_i) = a_i^2*Var(m) [common] + Var(n_i) [independent]
    result += f"\n{'channel':>10}{'a_i (mV)':>10}{'Var(A_i)':>12}{'a^2*Var(m)':>12}{'Var(n_i)':>12}{'% indep':>9}{'coupled':>9}\n"
    clamped_any = False
    for i, s in enumerate(SENSORS):
        vn = var_n[i]
        if vn < 0: # unphysical: a_i^2*Var(m) > Var(A_i)
            vn = 0.0  # catastrophic cancellation -> report as ~0
            clamped_any = True
        pct_indep = 100 * vn / var_A[i]
        flag = "yes" if is_coupled[i] else "NO"
        result += (f"{s:>10}{a_vec[i]:>10.3f}{var_A[i]:>12.4f}" f"{var_common[i]:>12.4f}{vn:>12.4f}{pct_indep:>8.1f}%{flag:>9}\n")
    if clamped_any:
        result += "  Var(n_i)=0 (clamped): a_i^2*Var(m) >= Var(A_i), i.e. ~100% common mode; independent noise is below this method's resolution.\n"

result += f"\nE[RMS(m)] over {NUM_TRIALS} trial(s) = {np.sqrt(np.nanmean(var_m_trials)):.4f}  (fractional LED intensity fluctuation)\n"

result += f"\n{'=' * 64}\nINTERPRETATION\n{'=' * 64}\n"
result += (
    "Here are the variables in the model V(t) = A_i(t)cos(2πf_c t + θ),  A_i(t) = a_i + a_i·m(t) + n_i(t):\n\n"
    "Measured / raw\n"
    "    V_i(t)      — the raw load voltage of diode i (the carrier sine you actually sample), mV.\n"
    "    f_c         — carrier frequency, ~287 Hz (LED drive tone). θ is its phase.\n\n"
    "Per-diode envelope (after I/Q demod & de-rotation)\n"
    "    A_i(t)      — demodulated envelope of diode i: the slowly-varying carrier amplitude, mV. "
    "This is what all the covariance math runs on.\n"
    "    a_i         — mean carrier amplitude of diode i (the constant part of A_i), mV. "
    "Represents the average optical power reaching that diode. ~8–18 mV now.\n\n"
    "The two things we're separating\n"
    "    m(t)        — LED's fractional intensity fluctuation, dimensionless, E[m]=0. "
    "Shared by all diodes (common mode). RMS(m) ≈ 1–2% = how much the brightness wobbles relative to its mean.\n"
    "    n_i(t)      — independent per-diode in-band noise (shot/thermal + local pickup), mV. "
    "Uncorrelated across diodes. Currently pinned at the electronics floor.\n\n"
    "Computed statistics\n"
    "    Var(m)      — variance of m, dimensionless. Estimated as Cov(A_i,A_j)/(a_i·a_j).\n"
    "    RMS(m)      — √Var(m), the fractional fluctuation (0.01 = 1%). E[RMS(m)] averages it over trials.\n"
    "    Var(A_i)    — total variance of diode i's envelope, mV² (diagonal of cov_A).\n"
    "    a_i²·Var(m) — the part of Var(A_i) explained by the shared LED (common mode), mV².\n"
    "    Var(n_i)    — remainder Var(A_i) − a_i²·Var(m) = independent noise, mV².\n"
    "    cov_A       — envelope covariance matrix, mV² (Var(A_i) on diagonal, shared fluctuation off-diagonal).\n"
    "    corr_A      — envelope correlation matrix, −1…1 (near 1 = tracking the same LED).\n"
    "    norm_cov    — cov_A / (a_i·a_j); off-diagonals are the per-pair Var(m) estimate.\n"
)

print(result)

data_to_save = [data[i] for i in range(NUM_TRIALS)]
file_names = [f"data_{PERIODS[i]}.csv" for i in range(NUM_TRIALS)]
save_run_data(FOLDER, data=data_to_save, filenames=file_names, sample_count=NUM_SAMPLES_TO_SAVE, summary=result)

A_i is the demodulated message of sensor i (LED FLUCTUATION + CIRCUIT NOISE + INTERFERENCE...)
a_i is the mean carrier amplitude of sensor i
Cov(A_i, A_j) = a_i * a_j * Var(m)  ->  Var(m) = Cov(A_i, A_j) / (a_i * a_j)
Var(n_i) = Var(A_i) - a_i^2 * Var(m)  (independent per-diode noise)

Across 4 sensors over 2 independent windows, Fs = 5000.00 Hz, Carrier Frequency = 930.00 Hz

TRIAL t1
LO frequency error df = +0.0163 Hz (ref channel: A2V-16_A3)


[1] COVARIANCE (mV^2) - diagonal = total fluctuations of each channel, off-diagonal = fluctuation shared between two channels
             A2V-16_A1   A2V-16_A2   A2V-16_A3   A2V-16_A4
 A2V-16_A1      0.4653      0.4973      0.5138      0.4999
 A2V-16_A2      0.4973      0.5787      0.5657      0.5525
 A2V-16_A3      0.5138      0.5657      0.6137      0.5744
 A2V-16_A4      0.4999      0.5525      0.5744      0.5614

[2] CORRELATION (-1 to 1)- near 1 = channels track togther (shared LED), 0 = independent
             A2V-16_A1   A2V-16_A2   A

WindowsPath('data/cross_sensor_variance_data/07_22_2026/run_15_56_13')

## Online RLS

In [ ]:
### CONFIG

NUM_SAMPLES_TO_SAVE = 20000 
FOLDER = "rls_filtered_data"
REF = "BPW34_1"
SETTLE_TAU = 7.0 

###

USABLE_SENSORS = [s for s in SENSORS if s != REF]
TRANSIENT = int(np.ceil(SETTLE_TAU / (np.pi * BANDWIDTH) * Fs)) 
TRANSIENT = int(np.ceil(TRANSIENT/1000) * 1000)  # round up to nearest 1000 samples

sos_bps = {s: signal.butter(4, [CARRIER_FREQ - 2, CARRIER_FREQ + 2], btype='bp', fs=Fs, output='sos') for s in SENSORS}
sos_lps = {s: signal.butter(4, 20, btype='low', fs=Fs, output='sos') for s in SENSORS}
zi_bps = {s: np.zeros((len(sos_bps[s]), 2)) for s in SENSORS} # type: ignore
zi_I  = {s: np.zeros((len(sos_lps[s]), 2)) for s in SENSORS} # type: ignore
zi_Q  = {s: np.zeros((len(sos_lps[s]), 2)) for s in SENSORS} # type: ignore
sample_count = 0

NTAPS = 1
filters  = {s: pa.filters.FilterRLS(n=NTAPS, mu=0.99999) for s in USABLE_SENSORS} # type: ignore
ref_hist = np.zeros(NTAPS)       # persistent rolling window of A[REF] across packets

def filter_pipeline(data):
    global sample_count, ref_hist
    N = SAMPLES_PER_PACKET
    t = np.arange(sample_count, sample_count + N) / Fs      # continuous LO phase
    lo_cos, lo_sin = np.cos(2*np.pi*CARRIER_FREQ*t), -np.sin(2*np.pi*CARRIER_FREQ*t)
    sample_count += N
 
    A = {}
    for s in SENSORS:
        v_mv = adc_to_mv(data[s])
        v_bp, zi_bps[s] = signal.sosfilt(sos_bps[s], v_mv, zi=zi_bps[s])
        I, zi_I[s] = signal.sosfilt(sos_lps[s], lo_cos*v_bp, zi=zi_I[s])
        Q, zi_Q[s] = signal.sosfilt(sos_lps[s], lo_sin*v_bp, zi=zi_Q[s])
        A[s] = 2.0 * np.sqrt(I**2 + Q**2)

    errors = {s: np.empty(N) for s in USABLE_SENSORS}
    ref = A[REF]
    for i in range(N):
        x = ref[i:i+1]                        # length-1 input window
        for s in USABLE_SENSORS:
            y = filters[s].predict(x)         # estimated common mode = w·A_REF
            filters[s].adapt(A[s][i], x)      # desired = sensor sample
            errors[s][i] = A[s][i] - y        # cleaned = sensor − common mode
    return errors

final_data = {s: [] for s in USABLE_SENSORS}

timeout = 0
sample_count = 0
if LOAD_FROM_FILE and reader is None:
    reader = read_from_file_realtime(LOAD_FILE_PATHS[0])
with serial_port():                      # Fix 1: guaranteed close (no-op when LOAD_FROM_FILE)
    if not LOAD_FROM_FILE:
        drain_to_latest(ser)
    while NUM_SAMPLES_TO_SAVE is None or sample_count < NUM_SAMPLES_TO_SAVE:
        if LOAD_FROM_FILE:
            data = reader()
            if data is None:  # reader is pacing itself against wall clock
                break
        else:
            data = try_read_realtime(ser)
            if data is not None:
                d = filter_pipeline(data)
                for s in USABLE_SENSORS:
                    final_data[s].append(d[s])
            sample_count += SAMPLES_PER_PACKET


sig = {s: np.concatenate(final_data[s])[TRANSIENT:] for s in USABLE_SENSORS}

save_run_data(FOLDER, data=[sig[s] for s in USABLE_SENSORS], filenames=[f"data_{s}.csv" for s in USABLE_SENSORS], sample_count=sample_count, summary=f"Sampling Rate = {Fs:.2f} Hz\nCarrier Frequency = {CARRIER_FREQ:.6f} Hz\n")


## Noise Formulas

In [ ]:
# Photodiode + TIA noise budget.
# Every source is a POWER spectral density [A^2/Hz]; multiply by the noise bandwidth
# and sqrt for an RMS current, then x |Z_f| to refer it to the TIA output voltage.

import numpy as np

# ============================================================
# TUNABLE VALUES 
# ============================================================
# -- physical constants --
Q   = 1.602176634e-19   # elementary charge [C]
KB  = 1.380649e-23      # Boltzmann constant [J/K]
T   = 300.0             # temperature [K]

# -- optical / photodiode --
RESPONSIVITY = 0.55     # detector responsivity [A/W]  (BPW34 ~0.4 in the red)
P_OPTICAL    = 2.5e-6   # optical power on the diode [W]  -> sets the photocurrent
I_DARK       = 1.5e-9   # dark current [A]  (BPW34, low reverse bias)
C_DIODE      = 25e-12   # junction capacitance [F]  (BPW34 ~70pF @ 0V, ~25pF reverse-biased)

# -- TIA --
R_F  = 1.0e6            # feedback resistor = transimpedance gain [ohm]
C_F  = 1.0e-12          # feedback capacitor (stability pole) [F]
R_SH = 1.0e9            # photodiode shunt resistance [ohm]

# -- op-amp input noise (from its datasheet) --
E_N  = 8.0e-9           # input voltage noise density [V/sqrt(Hz)]
I_N  = 5.0e-15          # input current noise density [A/sqrt(Hz)]
C_IN = 5.0e-12          # amplifier input capacitance [F]

# -- measurement band --
F_EVAL  = 287.0         # frequency the noise is evaluated at (your carrier) [Hz]
BAND_HZ = 4.0           # noise bandwidth of the demod (+/-2 Hz -> 4 Hz)

# ============================================================
# DERIVED OPERATING POINT
# ============================================================
I_photo = RESPONSIVITY * P_OPTICAL
I_total = I_photo + I_DARK
C_tot   = C_DIODE + C_IN                        # total input-node capacitance
w       = 2 * np.pi * F_EVAL

Z_f   = R_F / (1 + 1j * w * R_F * C_F)          # feedback impedance  R_f || C_f
Z_s   = R_SH / (1 + 1j * w * R_SH * C_tot)      # source impedance    R_sh || C_tot
NG    = abs(1 + Z_f / Z_s)                      # noise gain seen by e_n
absZf = abs(Z_f)                                # transimpedance at F_EVAL [ohm]
V_T   = KB * T / Q                              # thermal voltage [V]

sqrtB = np.sqrt(BAND_HZ)
pA = lambda x: f"{x * 1e12:8.3f} pA/rtHz"       # current density
nV = lambda x: f"{x * 1e9:8.2f} nV/rtHz"        # voltage density
uV = lambda x: f"{x * 1e6:8.3f} uV rms"         # rms voltage over BAND_HZ

contrib = {}   # name -> output voltage density [V/rtHz]

def report_current_source(name, i_dens):
    """A white current noise that flows through the feedback impedance."""
    v_dens = i_dens * absZf
    contrib[name] = v_dens
    print(f"  input-referred : {pA(i_dens)}")
    print(f"  output density : {nV(v_dens)}   (x |Z_f| = {absZf/1e6:.3f} MOhm)")
    print(f"  output rms     : {uV(v_dens * sqrtB)}")

print("=" * 60)
print("OPERATING POINT")
print("=" * 60)
print(f"  I_photo        = {I_photo*1e6:.4f} uA   (R * P_opt)")
print(f"  I_dark         = {I_DARK*1e9:.4f} nA")
print(f"  I_total        = {I_total*1e6:.4f} uA")
print(f"  V_out (DC)     = I_total * R_f = {I_total*R_F:.4f} V   (check vs supply rails)")
print(f"  C_tot          = {C_tot*1e12:.1f} pF   (C_diode + C_in)")
print(f"  |Z_f| @ {F_EVAL:.0f} Hz = {absZf/1e6:.4f} MOhm   (transimpedance at the carrier)")
print(f"  thermal voltage kT/q = {V_T*1e3:.2f} mV")

print("\n" + "=" * 60)
print("STEP 1 - SHOT NOISE   S = 2q*I_total   (scales as sqrt(I))")
print("=" * 60)
i_shot = np.sqrt(2 * Q * I_total)
report_current_source("shot", i_shot)

print("\n" + "=" * 60)
print("STEP 2 - DARK-CURRENT SHOT   S = 2q*I_dark   (subset of step 1)")
print("=" * 60)
i_dark = np.sqrt(2 * Q * I_DARK)
print(f"  input-referred : {pA(i_dark)}   (flows even with the LED off)")

print("\n" + "=" * 60)
print("STEP 3 - FEEDBACK-RESISTOR THERMAL   S = 4kT/R_f   (amplitude-independent floor)")
print("=" * 60)
i_rf = np.sqrt(4 * KB * T / R_F)
report_current_source("Rf_thermal", i_rf)

print("\n" + "=" * 60)
print("STEP 4 - OP-AMP CURRENT NOISE   S = i_n^2")
print("=" * 60)
report_current_source("opamp_in", I_N)

print("\n" + "=" * 60)
print("STEP 5 - OP-AMP VOLTAGE NOISE   e_n * noise_gain   (the 'e_n*C' term)")
print("=" * 60)
v_en = E_N * NG
contrib["opamp_en"] = v_en
f_zero = 1 / (2 * np.pi * R_F * C_tot)  
f_pole = 1 / (2 * np.pi * R_F * C_F)  
print(f"  noise gain @ {F_EVAL:.0f} Hz = {NG:.3f}   (rises toward 1 + C_tot/C_f = {1 + C_tot/C_F:.1f})")
print(f"  e_n*C corner   : zero {f_zero:.1f} Hz,  pole {f_pole/1e3:.1f} kHz")
print(f"  output density : {nV(v_en)}")
print(f"  output rms     : {uV(v_en * sqrtB)}")
print(f"  (equiv input current e_n*2*pi*f*C_tot = {pA(E_N * w * C_tot)})")

print("\n" + "=" * 60)
print("STEP 6 - TOTAL NOISE (quadrature sum of the output densities)")
print("=" * 60)
v_tot = np.sqrt(sum(v**2 for v in contrib.values()))
v_tot_rms = v_tot * sqrtB
i_tot_in  = v_tot / absZf                 # input-referred total current density
NEP = i_tot_in / RESPONSIVITY             # noise-equivalent power [W/rtHz]
print(f"  total output density = {nV(v_tot)}")
print(f"  total output rms     = {uV(v_tot_rms)}   over {BAND_HZ:.0f} Hz")
print(f"  input-referred       = {pA(i_tot_in)}")
print(f"  NEP                  = {NEP*1e15:.2f} fW/rtHz")
print("  breakdown (% of noise VARIANCE):")
for name, v in sorted(contrib.items(), key=lambda kv: -kv[1]):
    print(f"     {name:<12}: {100 * (v**2) / (v_tot**2):5.1f} %")

print("\n" + "=" * 60)
print("STEP 7 - SIGNAL & SNR")
print("=" * 60)
v_sig = I_photo * absZf
print(f"  signal output    = I_photo * |Z_f| = {v_sig*1e3:.3f} mV")
print(f"  SNR (power)      = {10*np.log10((v_sig/v_tot_rms)**2):.1f} dB")
print(f"  SNR (amplitude)  = {v_sig / v_tot_rms:.0f} x")

print("\n" + "=" * 60)
print("STEP 8 - REGIME:  shot-limited vs thermal(floor)-limited")
print("=" * 60)
shot_over_thermal = (2 * Q * I_total) / (4 * KB * T / R_F)   # PSD ratio
print(f"  shot / Rf-thermal (power) = {shot_over_thermal:.2f}")
print(f"  crossover target : I_total * R_f > 2kT/q = {2*V_T*1e3:.1f} mV")
print(f"  you have         : I_total * R_f = {I_total*R_F*1e3:.1f} mV")
if I_total * R_F > 2 * V_T:
    print("  -> SHOT-LIMITED: Var(n) scales with optical power.")
else:
    print("  -> THERMAL/ELECTRONIC-LIMITED: Var(n) is fixed (the additive floor you see now).")
    print(f"     raise I_photo or R_f by {2*V_T/(I_total*R_F):.1f}x to cross into shot-limited.")


## Averaging

In [ ]:
### CONFIG

NUM_SAMPLES_TO_SAVE = 210000  # set to None to run until interrupted
NUM_TRIALS = 200000               # fingerprint reads; trial length is derived from the record
SETTLE_TAU = 7.0              # filter time constants to drop at the start (7*tau ~= -60 dB ring-down)
FOLDER = "averaging_data"
DEMOD_LP_HZ = 20              # Hz, low-pass cutoff for the demodulated envelope
VERBOSE = False

###

sos_lps = {s: signal.butter(4, DEMOD_LP_HZ, btype='low', fs=Fs, output='sos') for s in SENSORS}
zi_I   = {s: np.zeros((len(sos_lps[s]), 2)) for s in SENSORS}  # type: ignore
zi_Q   = {s: np.zeros((len(sos_lps[s]), 2)) for s in SENSORS}  # type: ignore

TRANSIENT = int(np.ceil(SETTLE_TAU / (np.pi * DEMOD_LP_HZ) * Fs))


def filter_pipeline(data):
    # Returns the COMPLEX baseband Z = I + jQ.
    global sample_count
    N = SAMPLES_PER_PACKET
    t = np.arange(sample_count, sample_count + N) / Fs
    lo_cos, lo_sin = np.cos(2*np.pi*CARRIER_FREQ*t), -np.sin(2*np.pi*CARRIER_FREQ*t)

    Z = {}
    for s in SENSORS:
        v_mv = adc_to_mv(data[s])
        I, zi_I[s] = signal.sosfilt(sos_lps[s], lo_cos * v_mv, zi=zi_I[s])
        Q, zi_Q[s] = signal.sosfilt(sos_lps[s], lo_sin * v_mv, zi=zi_Q[s])
        Z[s] = I + 1j * Q
    return Z


sample_count = 0
timeouts = 0
final_data = {s: [] for s in SENSORS}
raw_data = []

if LOAD_FROM_FILE:
    data = read_from_file_instant(LOAD_FILE_PATHS[0])
    if data is not None:
        z = filter_pipeline(data)
        for s in SENSORS:
            final_data[s].append(z[s])
else:
    with serial_port():           # Fix 1: guaranteed close even on interrupt
        drain_to_latest(ser)
        while NUM_SAMPLES_TO_SAVE is None or sample_count < NUM_SAMPLES_TO_SAVE:
            data = try_read_realtime(ser)
            if data is None:
                timeouts += 1
                if timeouts > 20:
                    print("serial timed out repeatedly - stopping early")
                    break
                continue
            timeouts = 0
            raw_data.append(data)

            z = filter_pipeline(data)   # advances sample_count -- do NOT increment it again here,
            for s in SENSORS:           # or the LO time vector skips a packet of phase each packet
                final_data[s].append(z[s])
            sample_count += SAMPLES_PER_PACKET




z_all = {s: np.concatenate(final_data[s])[TRANSIENT:] for s in SENSORS}
n     = len(z_all[SENSORS[0]])
t_all = np.arange(TRANSIENT, TRANSIENT + n) / Fs

# de-rotate
ref = max(SENSORS, key=lambda s: np.abs(z_all[s]).mean())
phi = np.unwrap(np.angle(z_all[ref]))
slope, offset = np.polyfit(t_all, phi, 1)
freq_err = slope / (2 * np.pi)                                  # Hz
sig = {s: z_all[s] * np.exp(-1j * (slope * t_all + offset)) for s in SENSORS}

# split the settled record into NUM_TRIALS equal trials -> one fingerprint read each
TRIAL_SIZE = n // NUM_TRIALS
if TRIAL_SIZE == 0:
    raise RuntimeError(f"only {n} samples after settling - not enough for {NUM_TRIALS} trial(s)")

a_hat, bits = {}, {}
for g in range(NUM_TRIALS):
    sl = slice(g * TRIAL_SIZE, (g + 1) * TRIAL_SIZE)
    a_hat[g] = {s: 2 * np.abs(np.mean(sig[s][sl])) for s in SENSORS}
    bits[g] = [1 if a_hat[g][SENSORS[i]] > a_hat[g][SENSORS[j]] else 0
               for i in range(len(SENSORS)) for j in range(i + 1, len(SENSORS))]

# --- report ------------------------------------------------------------------------
result  = f"LO frequency error   df = {freq_err:+.4f} Hz  (large |df| -> re-run Setup / refine CARRIER_FREQ)\n"
result += f"{NUM_TRIALS} trials x {TRIAL_SIZE} samples ({TRIAL_SIZE/Fs:.2f} s each), {n} samples after settling ({TRANSIENT} dropped, {TRANSIENT/Fs:.3f} s)\n\n"

if VERBOSE:
    for g in range(NUM_TRIALS):
        means = "  ".join(f"{s}={a_hat[g][s]:7.3f}" for s in SENSORS)
        result += f"Trial {g+1:2d}: {means}   bits={bits[g]}\n"

result += "\nPer-sensor drift across trials (mV) - the floor the surface fingerprint must beat:\n"
for s in SENSORS:
    v = np.array([a_hat[g][s] for g in range(NUM_TRIALS)])
    result += f"  {s:9s} mean={v.mean():7.3f}  std={v.std():7.4f}  CV={100*v.std()/abs(v.mean()):5.2f}%\n"

result += "\nPairwise ratio drift (common mode cancels in the ratio):\n"
for i in range(len(SENSORS)):
    for j in range(i + 1, len(SENSORS)):
        r = np.array([a_hat[g][SENSORS[i]] / a_hat[g][SENSORS[j]] for g in range(NUM_TRIALS)])
        result += f"  {SENSORS[i]:9s}/{SENSORS[j]:9s} mean={r.mean():7.4f}  std={r.std():7.4f}  CV={100*r.std()/abs(r.mean()):5.2f}%\n"

allbits = np.array([bits[g] for g in range(NUM_TRIALS)])
uniq, counts = np.unique(allbits, axis=0, return_counts=True)
modal_bitstring = [int(x) for x in uniq[counts.argmax()]]
result += f"nBit stability: {counts.max()}/{NUM_TRIALS} trials gave the bitstring {' '.join(map(str, modal_bitstring))}\n"
result += f"{len(uniq)} distinct bitstring(s) seen (1 = perfectly reproducible)\n"

print(result)

save_run_data(FOLDER, data=[np.concatenate(raw_data)], filenames=["data.csv"], sample_count=NUM_SAMPLES_TO_SAVE, summary=result)


LO frequency error   df = +0.0163 Hz  (large |df| -> re-run Setup / refine CARRIER_FREQ)
200000 trials x 1 samples (0.00 s each), 209442 samples after settling (558 dropped, 0.112 s)


Per-sensor drift across trials (mV) - the floor the surface fingerprint must beat:
  A2V-16_A1 mean=967.273  std= 0.5851  CV= 0.06%
  A2V-16_A2 mean=1059.314  std= 0.7086  CV= 0.07%
  A2V-16_A3 mean=1112.457  std= 0.6971  CV= 0.06%
  A2V-16_A4 mean=1073.115  std= 0.4751  CV= 0.04%

Pairwise ratio drift (common mode cancels in the ratio):
  A2V-16_A1/A2V-16_A2 mean= 0.9131  std= 0.0006  CV= 0.07%
  A2V-16_A1/A2V-16_A3 mean= 0.8695  std= 0.0006  CV= 0.06%
  A2V-16_A1/A2V-16_A4 mean= 0.9014  std= 0.0004  CV= 0.05%
  A2V-16_A2/A2V-16_A3 mean= 0.9522  std= 0.0007  CV= 0.07%
  A2V-16_A2/A2V-16_A4 mean= 0.9871  std= 0.0006  CV= 0.06%
  A2V-16_A3/A2V-16_A4 mean= 1.0367  std= 0.0005  CV= 0.05%
nBit stability: 200000/200000 trials gave the bitstring 0 0 0 0 0 1
1 distinct bitstring(s) seen (1 = perfectly reproduci

WindowsPath('data/averaging_data/07_22_2026/run_15_58_28')

# Carrier Check

In [ ]:
### CONFIG

F0        = CARRIER_FREQ   # carrier fundamental (Hz), estimated in Setup
FS        = Fs             # sample rate (Hz), estimated in Setup
PWM_DUTY  = 0.50           # 0.50 => even harmonics vanish (phase2 ruling: keep 50%)
GUARD_HZ  = 2.0            # demod half-band around the carrier (±2 Hz working band)
DBC_FLOOR = -40.0          # ignore harmonics weaker than this vs. the fundamental
MAX_ORDER = 4000           # sweep depth (comb is endless; DBC_FLOOR does the culling)

###

nyq = FS / 2.0

def fold(f, fs):
    """Alias a real tone at frequency f into [0, fs/2]."""
    f = np.abs(f) % fs
    return fs - f if f > fs / 2 else f

def pwm_dbc(k, duty):
    """k-th harmonic of a duty-D pulse train, in dBc vs. the fundamental.
    np.sinc is the normalized sinc: |a_k| ∝ |sinc(k*duty)|."""
    ak, a1 = abs(np.sinc(k * duty)), abs(np.sinc(duty))
    return -np.inf if ak == 0 else 20 * np.log10(ak / a1)

# --- sweep the comb (harmonics k >= 2) ----------------------------------------
rows = []
for k in range(2, MAX_ORDER + 1):
    dbc = pwm_dbc(k, PWM_DUTY)
    if dbc < DBC_FLOOR:          # too weak to matter
        continue
    alias = fold(k * F0, FS)
    rows.append((k, k * F0, alias, dbc, abs(alias - F0)))

# --- verdicts -----------------------------------------------------------------
print(f"Fs = {FS:.3f} Hz   Nyquist = {nyq:.3f} Hz   f0 = {F0:.3f} Hz   duty = {PWM_DUTY:.0%}")
print(f"guard = ±{GUARD_HZ} Hz   dBc floor = {DBC_FLOOR}   orders swept = 2..{MAX_ORDER}\n")

if F0 < nyq:
    print(f"[OK]   fundamental below Nyquist  ({F0:.3f} < {nyq:.3f} Hz)")
else:
    print(f"[FAIL] fundamental {F0:.3f} Hz is at/above Nyquist {nyq:.3f} Hz — undersampled!")

collisions = [r for r in rows if r[4] <= GUARD_HZ]
if collisions:
    print(f"\n[FAIL] {len(collisions)} harmonic(s) alias into the ±{GUARD_HZ} Hz carrier band:")
    for k, fk, al, dbc, d in collisions:
        print(f"   k={k:<5d} {fk:11.1f} Hz -> {al:8.3f} Hz  ({dbc:+.1f} dBc, {d:.3f} Hz from f0)")
else:
    k, fk, al, dbc, d = min(rows, key=lambda r: r[4])
    print(f"\n[OK]   no harmonic within ±{GUARD_HZ} Hz of the carrier.")
    print(f"       nearest approach: k={k} ({dbc:+.1f} dBc) folds to {al:.3f} Hz, {d:.3f} Hz away.")

# --- spur table ---------------------------------------------------------------
print("\nspur table (significant harmonics):")
print(f"   {'k':>5}  {'harmonic Hz':>12}  {'alias Hz':>10}  {'dBc':>7}")
for k, fk, al, dbc, d in rows:
    print(f"   {k:5d}  {fk:12.1f}  {al:10.3f}  {dbc:+7.1f}")


## Noise Floor Check

In [ ]:
### CONFIG

NUM_SAMPLES_TO_SAVE = 150000       # samples to load/capture (~30 s @ 5 kHz for a stable PSD)
NFFT                = 8192         # Welch segment length -> bin width = Fs/NFFT
BAND_HALFWIDTHS     = [2, 5, 10, 20]   # Hz, swept per band; a true floor is bandwidth-invariant (density)
DEMOD_NOISE_BW      = 4.0          # Hz, analysis band (±2 Hz bandpass) the floor is scaled to for Var(n)
CARRIER_HW          = 4.0          # Hz, half-width to integrate the carrier tone (signal power for the SNR)
GUARD_HZ            = 3.0          # keep bands this far from every known comb line
LOW_CUT_HZ         = 20.0         # ignore DC / drift / 1-f below this
PEAK_DB            = 6.0          # a band is contaminated if any bin exceeds its median by > this
DUTY               = 0.50         # PWM duty, to exclude (future) square-wave harmonic aliases too
HARMONIC_DBC       = -50.0        # only exclude carrier harmonics stronger than this vs the fundamental
FORCE_BANDS_HZ     = []           # extra band centers to check even if the auto-scan skips them
FOLDER             = "noise_floor_data"

###

nyq    = Fs / 2.0
bin_hz = Fs / NFFT

# one long record, same read path as the other analysis cells
data   = read_from_file_instant(LOAD_FILE_PATHS[0]) if LOAD_FROM_FILE else read_data_realtime(NUM_SAMPLES_TO_SAVE)
data   = data[:NUM_SAMPLES_TO_SAVE]
n_samp = len(data)
nper   = min(NFFT, n_samp)

# Welch PSD per sensor (mV^2/Hz)
psd = {}
for s in SENSORS:
    freqs, psd[s] = welch(adc_to_mv(data[s]), fs=Fs, nperseg=nper, noverlap=nper // 2,
                          window='hann', detrend='constant')

# ---- forbidden comb: every line a "carrier-free" band must avoid -------------
def fold(fr):                                 # alias a tone into [0, nyq]
    fr = abs(fr) % Fs
    return Fs - fr if fr > nyq else fr

def pwm_dbc(k):                               # k-th harmonic of a duty-DUTY pulse train, dBc vs fundamental
    ak, a1 = abs(np.sinc(k * DUTY)), abs(np.sinc(DUTY))
    return -np.inf if ak == 0 else 20 * np.log10(ak / a1)

forbidden = [float(CARRIER_FREQ)]                                       # carrier fundamental
for k in range(2, 200):                                                 # + (future PWM) harmonic aliases
    if pwm_dbc(k) >= HARMONIC_DBC:
        forbidden.append(float(fold(k * CARRIER_FREQ)))
forbidden += [float(500 * m) for m in range(1, int(nyq // 500) + 1)]    # 1 kHz MCU event -> 500/1000/... lines
forbidden += [float(50  * m) for m in range(1, int(nyq // 50)  + 1)]    # serial packet comb
forbidden  = np.array(sorted({round(x, 3) for x in forbidden if x > 0}))

FORBIDDEN_RANGES = [(400.0, 460.0)]           # wandering non-clock-locked interferer (hardware.md)
def in_forbidden_range(c):
    return any(rlo <= c <= rhi for rlo, rhi in FORBIDDEN_RANGES)

# ---- candidate bands = midpoints of the widest clean gaps in the comb --------
edges = np.array(sorted({LOW_CUT_HZ, nyq} | set(forbidden[(forbidden > LOW_CUT_HZ) & (forbidden < nyq)])))
cand = []
for a, b in zip(edges[:-1], edges[1:]):
    c = 0.5 * (a + b)
    hw_avail = 0.5 * (b - a) - GUARD_HZ
    if hw_avail >= min(BAND_HALFWIDTHS) and not in_forbidden_range(c):
        cand.append((round(c, 1), min(hw_avail, max(BAND_HALFWIDTHS))))
for x in FORCE_BANDS_HZ:
    cand.append((float(x), max(BAND_HALFWIDTHS)))
cand = sorted(set(cand))

# ---- measure the floor in each band, per sensor ------------------------------
def band_stats(pxx, center, hw):
    seg = pxx[(freqs >= center - hw) & (freqs <= center + hw)]
    if len(seg) < 4:
        return None
    return float(np.median(seg)), 10 * np.log10(seg.max() / np.median(seg))   # density mV^2/Hz, worst-bin dB

records = {s: [] for s in SENSORS}
for center, hw_band in cand:
    hw_top = min(hw_band, max(BAND_HALFWIDTHS))
    for s in SENSORS:
        dens_by_hw = []
        for hw in BAND_HALFWIDTHS:
            if hw > hw_band:
                continue
            st = band_stats(psd[s], center, hw)
            if st is not None:
                dens_by_hw.append(st[0])
        if not dens_by_hw:
            continue
        st_top = band_stats(psd[s], center, hw_top)   # flatness at the widest usable halfwidth
        if st_top is None:
            continue
        worst_db = st_top[1]
        records[s].append(dict(center=center,
                               density=float(np.median(dens_by_hw)),
                               worst_db=worst_db,
                               bw_ratio=max(dens_by_hw) / min(dens_by_hw),   # ~1 => bandwidth-invariant
                               clean=(worst_db <= PEAK_DB)))


# ---- report ------------------------------------------------------------------
# consensus floor per sensor = median density over bands that are BOTH flat and
# bandwidth-invariant (i.e. genuinely empty), scaled to the demod band.
summary = {}
for s in SENSORS:
    good = [r for r in records[s] if r['clean'] and r['bw_ratio'] < 2.0]
    dens = np.array(sorted(r['density'] for r in good))
    if len(dens):
        floor = float(np.median(dens))
        q1, q3 = np.percentile(dens, [25, 75])
        summary[s] = dict(floor=floor, var=floor * DEMOD_NOISE_BW,
                          rms_uv=np.sqrt(floor * DEMOD_NOISE_BW) * 1000,
                          n_good=len(good), n_rej=len(records[s]) - len(good),
                          spread=(q3 / q1 if q1 > 0 else np.inf),
                          examples=[r['center'] for r in sorted(good, key=lambda r: r['worst_db'])[:3]])
    else:
        summary[s] = None

quiet = min((d['floor'] for d in summary.values() if d), default=None)

# ---- carrier level & per-channel SNR ----------------------------------------
# Signal = the carrier tone at CARRIER_FREQ (integrate its PSD peak -> a^2/2).
# Noise  = the channel's own floor scaled to DEMOD_NOISE_BW, i.e. Var(n).
# SNR is reported in that demod band (the working band the cross-sensor cell
# uses) as carrier power / noise power. This is the raw per-channel carrier SNR,
# not the inter-diode PUF comparison SNR (that lives in the cross-sensor cell).
def carrier_power(pxx, floor_density):        # integrated tone power at the carrier, mV^2
    m = (freqs >= CARRIER_FREQ - CARRIER_HW) & (freqs <= CARRIER_FREQ + CARRIER_HW)
    if m.sum() < 3:
        return None
    return float(np.trapezoid(np.clip(pxx[m] - floor_density, 0.0, None), freqs[m]))

for s in SENSORS:
    d = summary[s]
    if d is None:
        continue
    p_car = carrier_power(psd[s], d['floor'])          # mV^2, tone power = a_i^2 / 2
    if p_car is None or p_car <= 0:
        d['a_mv'], d['snr_db'] = None, None
        continue
    d['a_mv']   = float(np.sqrt(2.0 * p_car))          # carrier amplitude a_i, mV
    d['snr_db'] = float(10.0 * np.log10(p_car / d['var']))   # 10log10( (a^2/2) / Var(n) ), demod band

# ---- readable summary (printed, and saved as summary.txt) --------------------
result  = "=" * 68 + "\n"
result += " NOISE FLOOR  —  the noise each photodiode adds on its own\n"
result += "=" * 68 + "\n"
result += "Extra electronic/pickup noise per channel, measured only where no\n"
result += "carrier or interference lives, then scaled to the analysis band.\n"
result += f"Record : {n_samp} samples ({n_samp/Fs:.0f} s)   Fs = {Fs:.0f} Hz   carrier = {CARRIER_FREQ:.1f} Hz\n"
result += f"Scaled : to the {DEMOD_NOISE_BW:.0f} Hz demod window (same band the cross-sensor cell uses)\n\n"

result += (f"  {'sensor':<9}{'carrier mV':>12}{'noise uV rms':>13}{'SNR dB':>9}"
           f"{'Var(n) mV^2':>15}{'vs quietest':>13}{'consistency':>13}\n")
result += "  " + "-" * 84 + "\n"
for s in SENSORS:
    d = summary[s]
    if d is None:
        result += f"  {s:<9}{'— no clean band found —':>54}\n"
        continue
    tag     = "consistent" if d['spread'] < 2 else "varies*"
    a_str   = f"{d['a_mv']:.3f}"   if d.get('a_mv')   is not None else "—"
    snr_str = f"{d['snr_db']:.1f}" if d.get('snr_db') is not None else "—"
    result += (f"  {s:<9}{a_str:>12}{d['rms_uv']:>13.1f}{snr_str:>9}"
               f"{d['var']:>15.2e}{d['floor']/quiet:>12.1f}x{tag:>13}\n")

result += "\nHow to read this\n"
result += "  carrier mV    carrier amplitude a_i, from integrating the PSD peak at the\n"
result += "                carrier (tone power = a_i^2/2). This is the wanted signal.\n"
result += "  noise uV rms  RMS of the channel's own noise over the demod band, in µV.\n"
result += "                It is sqrt(Var(n)) — a standard deviation (zero-mean), not a\n"
result += "                peak and not a per-√Hz density.\n"
result += "  SNR dB        carrier power over noise power in the demod band,\n"
result += "                10·log10((a_i^2/2)/Var(n)) — i.e. carrier-RMS / noise-RMS in dB.\n"
result += "                Per-channel carrier SNR, not the inter-diode PUF comparison.\n"
result += "  Var(n) mV^2   the same noise as a variance — it IS the Var(n_i) the\n"
result += "                cross-sensor cell prints for this diode.\n"
result += "  vs quietest   how many times noisier than the best channel.\n"
result += "  consistency   'consistent' = same floor everywhere; 'varies*' = the floor\n"
result += "                changes with frequency (usually a 1/f rise down low), so read\n"
result += "                the number as a typical mid-band value, not an exact constant.\n"

result += "\nWhere each number came from\n"
for s in SENSORS:
    d = summary[s]
    if d is None:
        continue
    ex = ", ".join(f"{c:.0f}" for c in d['examples'])
    result += (f"  {s:<9} averaged over {d['n_good']} clean bands "
               f"(e.g. {ex} Hz);  {d['n_rej']} rejected as too near a spur\n")
result += "  full per-band table saved alongside as summary_verbose.txt\n"

# ---- verbose per-band table (saved to its own file, never printed) -----------
verbose  = "FULL PER-BAND NOISE-FLOOR TABLE\n"
verbose += f"Fs = {Fs:.2f} Hz  carrier = {CARRIER_FREQ:.3f} Hz  Nyquist = {nyq:.1f} Hz\n"
verbose += f"record = {n_samp} samples ({n_samp/Fs:.1f} s)  Welch NFFT = {nper}  bin = {bin_hz:.3f} Hz\n"
verbose += f"candidate carrier-free bands = {len(cand)}  (guard ±{GUARD_HZ} Hz from every comb line)\n"
verbose += f"'clean' = no bin > band-median + {PEAK_DB} dB;  bw_ratio = density spread across halfwidths {BAND_HALFWIDTHS}\n"
for s in SENSORS:
    verbose += f"\n{'=' * 72}\n{s}\n{'=' * 72}\n"
    verbose += f"{'center Hz':>10}{'density mV^2/Hz':>18}{'worst dB':>10}{'bw_ratio':>10}{'clean':>8}\n"
    for r in sorted(records[s], key=lambda r: r['density']):
        verbose += (f"{r['center']:>10.1f}{r['density']:>18.3e}{r['worst_db']:>10.1f}"
                    f"{r['bw_ratio']:>10.2f}{('yes' if r['clean'] else 'NO*'):>8}\n")
    d = summary[s]
    if d:
        verbose += (f"  consensus floor = {d['floor']:.3e} mV^2/Hz over {d['n_good']} clean bands  ->  "
                    f"Var(n) = {d['var']:.4f} mV^2  (RMS {d['rms_uv']:.1f} uV)\n")
        if d.get('a_mv') is not None:
            verbose += (f"  carrier a = {d['a_mv']:.3f} mV  ->  SNR = {d['snr_db']:.1f} dB "
                        f"in the {DEMOD_NOISE_BW:.0f} Hz demod band\n")

print(result)
save_dir = save_run_data(FOLDER, data=[data], filenames=["data.csv"], sample_count=NUM_SAMPLES_TO_SAVE, summary=result)
save_summary(save_dir, verbose, "summary_verbose.txt")


 NOISE FLOOR  —  the noise each photodiode adds on its own
Extra electronic/pickup noise per channel, measured only where no
carrier or interference lives, then scaled to the analysis band.
Record : 150000 samples (30 s)   Fs = 5000 Hz   carrier = 930.0 Hz
Scaled : to the 4 Hz demod window (same band the cross-sensor cell uses)

  sensor     carrier mV noise uV rms   SNR dB    Var(n) mV^2  vs quietest  consistency
  ------------------------------------------------------------------------------------
  A2V-16_A1     798.502         99.4     75.1       9.88e-03         2.3x   consistent
  A2V-16_A2     793.619        126.7     72.9       1.61e-02         3.8x   consistent
  A2V-16_A3     764.989        118.4     73.2       1.40e-02         3.3x   consistent
  A2V-16_A4     811.571         65.0     78.9       4.22e-03         1.0x   consistent

How to read this
  carrier mV    carrier amplitude a_i, from integrating the PSD peak at the
                carrier (tone power = a_i^2/2). This 

WindowsPath('data/noise_floor_data/07_22_2026/run_15_37_56/summary_verbose.txt')